In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc matplotlib numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Estimasi energi ground-state rantai Heisenberg dengan VQE

import TutorialFeedback from '@site/src/components/TutorialFeedback';





*Perkiraan penggunaan: 37 menit pada prosesor Heron (CATATAN: Ini hanya perkiraan. Waktu eksekusi aktualmu bisa berbeda.)*
## Hasil belajar
Setelah menyelesaikan tutorial ini, kamu diharapkan dapat memahami informasi berikut:
- Cara memodelkan rantai spin Heisenberg sebagai Hamiltonian kuantum menggunakan Qiskit
- Cara menggunakan optimizer SPSA untuk mengestimasi energi ground-state suatu sistem kuantum
- Cara mengeksekusi alur kerja variasional pada hardware kuantum IBM&reg; menggunakan primitif dan sesi Qiskit Runtime

## Prasyarat
Disarankan untuk terlebih dahulu mengenal topik-topik berikut:
- [Dasar-dasar informasi kuantum](/learning/courses/basics-of-quantum-information)
- [Pengantar pola Qiskit](/guides/intro-to-patterns)
- [Desain algoritma variasional](/learning/courses/variational-algorithm-design)

## Latar Belakang
Rantai spin Heisenberg adalah salah satu model yang paling banyak dipelajari dalam fisika materi terkondensasi dan kemagnetan kuantum. Model ini menggambarkan kisi satu dimensi dari spin kuantum yang saling berinteraksi, di mana spin-spin yang berdekatan saling terhubung melalui interaksi pertukaran. Hamiltonian untuk model Heisenberg isotropik dengan medan magnet eksternal diberikan oleh:

$$H = \sum_{\langle i,j \rangle} \left( J_x X_i X_j + J_y Y_i Y_j + J_z Z_i Z_j \right) + \sum_{i} h_i Z_i,$$

di mana $X_i$, $Y_i$, dan $Z_i$ adalah operator Pauli yang bekerja pada situs $i$, jumlah $\langle i,j \rangle$ berjalan atas pasangan tetangga terdekat, $J_x = J_y = J_z = 0.5$ adalah konstanta kopling pertukaran (isotropik dalam tutorial ini), dan $h_i$ mewakili medan magnet eksternal yang bergantung pada situs. Dalam tutorial ini, nilai medan magnet diambil secara acak dari interval $[-1, 1]$. Perlu diperhatikan bahwa dalam implementasi di bawah ini, himpunan pasangan "tetangga terdekat" ditentukan oleh kopling native backend hardware di antara $N$ qubit pertama, yang mungkin tidak membentuk rantai linear yang ketat tergantung pada topologi perangkat.

Memahami energi ground-state dari Hamiltonian ini sangat penting dalam fisika. State dasar mengkodekan informasi tentang transisi fase kuantum, struktur keterikatan (entanglement), dan pemesanan magnetik. Secara klasik, menghitung energi ground-state yang tepat menjadi sulit dilakukan seiring bertambahnya jumlah spin, karena dimensi ruang Hilbert bertumbuh secara eksponensial sebagai $2^N$ untuk $N$ spin. Hal ini menjadikannya kandidat alami untuk simulasi kuantum.

Variational Quantum Eigensolver (VQE) adalah algoritma hibrida kuantum-klasik yang dirancang untuk mengestimasi energi ground-state dari suatu Hamiltonian. Algoritma ini bekerja dengan menyiapkan state kuantum berparameter $|\psi(\theta)\rangle$ (disebut ansatz) pada komputer kuantum dan mengukur nilai ekspektasi $\langle \psi(\theta) | H | \psi(\theta) \rangle$. Optimizer klasik kemudian secara iteratif menyesuaikan parameter $\theta$ untuk meminimalkan energi ini, memanfaatkan prinsip variasional yang menjamin bahwa energi yang terukur selalu merupakan batas atas dari energi ground-state yang sebenarnya.

Dalam tutorial ini, kita menggunakan ansatz `efficient_su2` dari library sirkuit Qiskit, yang membangun lapisan-lapisan rotasi qubit tunggal dan gerbang entangling. Optimasi dilakukan menggunakan algoritma Simultaneous Perturbation Stochastic Approximation (SPSA), yang cocok untuk hardware kuantum yang berisik karena mengestimasi gradien hanya dengan dua evaluasi fungsi per iterasi terlepas dari jumlah parameter.
## Persyaratan
Sebelum memulai tutorial ini, pastikan kamu sudah menginstal hal-hal berikut:

* Qiskit SDK v2.0 atau lebih baru, dengan dukungan [visualisasi](https://docs.quantum.ibm.com/api/qiskit/visualization)
* Qiskit Runtime v0.44 atau lebih baru (`pip install qiskit-ibm-runtime`)
## Setup

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Sequence

from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import BaseEstimatorV2
from qiskit.circuit.library import XGate
from qiskit.circuit.library import efficient_su2
from qiskit.transpiler import PassManager
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit.transpiler.passes.scheduling import (
    ALAPScheduleAnalysis,
    PadDynamicalDecoupling,
)
from qiskit_ibm_runtime import QiskitRuntimeService, Session, EstimatorV2


def visualize_results(results):
    plt.plot(results["cost_history"], lw=2)
    plt.xlabel("Number of function evaluations")
    plt.ylabel("Energy")
    plt.show()

## Contoh skala kecil

Dalam bagian ini, kita akan membahas setiap langkah dari pola Qiskit dalam skala kecil, menjelaskan komponen-komponen kunci saat kita membangun alur kerja tersebut.

### Langkah 1: Memetakan input klasik ke masalah kuantum

* Input: Jumlah spin
* Output: Ansatz dan Hamiltonian yang memodelkan rantai Heisenberg

Buat ansatz dan Hamiltonian yang memodelkan rantai Heisenberg 10-spin. Dalam langkah ini, kita akan membangun Hamiltonian Heisenberg 10-spin berdasarkan peta kopling backend yang paling tidak sibuk dan menyiapkan ansatz `efficient_su2`.

In [2]:
num_spins = 10
ansatz = efficient_su2(num_qubits=num_spins, reps=2)

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, min_num_qubits=num_spins, simulator=False
)

coupling = backend.target.build_coupling_map()
reduced_coupling = coupling.reduce(list(range(num_spins)))

edge_list = reduced_coupling.graph.edge_list()
ham_list = []

for edge in edge_list:
    ham_list.append(("ZZ", edge, 0.5))
    ham_list.append(("YY", edge, 0.5))
    ham_list.append(("XX", edge, 0.5))

for qubit in reduced_coupling.physical_qubits:
    ham_list.append(("Z", [qubit], np.random.random() * 2 - 1))

hamiltonian = SparsePauliOp.from_sparse_list(ham_list, num_qubits=num_spins)

ansatz.draw("mpl", style="iqp")

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/7e8d2f10-f1d6-4ec2-bac9-9db23499c9e1-0.avif)

### Langkah 2: Mengoptimalkan masalah untuk eksekusi hardware kuantum
* Input: Circuit abstrak, observable
* Output: Target Circuit dan observable, dioptimalkan untuk QPU yang dipilih

Gunakan fungsi `generate_preset_pass_manager` dari Qiskit untuk secara otomatis menghasilkan rutinitas optimasi untuk Circuit kita terhadap QPU yang dipilih. Kita pilih `optimization_level=3`, yang memberikan tingkat optimasi tertinggi dari preset pass manager. Kita juga menyertakan scheduling pass `ALAPScheduleAnalysis` dan `PadDynamicalDecoupling` untuk menekan kesalahan dekoherensi.

In [3]:
target = backend.target
pm = generate_preset_pass_manager(optimization_level=3, target=target)
pm.scheduling = PassManager(
    [
        ALAPScheduleAnalysis(durations=target.durations()),
        PadDynamicalDecoupling(
            durations=target.durations(),
            dd_sequence=[XGate(), XGate()],
            pulse_alignment=target.pulse_alignment,
        ),
    ]
)
isa_ansatz = pm.run(ansatz)
isa_observable = hamiltonian.apply_layout(isa_ansatz.layout)
isa_ansatz.draw("mpl", scale=0.6, style="iqp", fold=-1, idle_wires=False)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/tutorials/spin-chain-vqe/extracted-outputs/a0a5f1c8-5c31-4d9f-ae81-37bd67271d44-0.avif)

### Langkah 3: Eksekusi menggunakan primitif Qiskit
* Input: Target Circuit dan observable
* Output: Hasil optimasi

Minimalkan estimasi energi ground-state sistem dengan mengoptimalkan parameter Circuit. Gunakan primitif `Estimator` dari Qiskit Runtime untuk mengevaluasi fungsi cost selama optimasi.

Karena kita sudah mengoptimalkan Circuit untuk backend di Langkah 2, kita bisa menghindari transpilasi di server Runtime dengan mengatur `skip_transpilation=True` dan melewatkan Circuit yang sudah dioptimalkan. Untuk demo ini, kita akan menjalankan pada QPU menggunakan primitif `qiskit-ibm-runtime`. Untuk menjalankan dengan primitif berbasis statevector `qiskit`, ganti blok kode yang menggunakan primitif Qiskit Runtime dengan blok yang dikomentari.

Dalam tutorial ini kita menggunakan Simultaneous Perturbation Stochastic Approximation (SPSA), yang merupakan optimizer berbasis gradien. Berikut ini kita berikan pengenalan singkat tentangnya, dan menyediakan kode untuk mengimplementasikan SPSA menggunakan Qiskit v2.0.
### Memperkenalkan SPSA
Simultaneous Perturbation Stochastic Approximation (SPSA) [\[1\]](#references) adalah algoritma optimasi yang memperkirakan seluruh vektor gradien hanya menggunakan dua pemanggilan fungsi di setiap iterasi. Misalkan $f:\mathbb{R}^p\rightarrow \mathbb{R}$ adalah fungsi cost dengan $p$ parameter yang akan dioptimalkan, dan $x_i\in \mathbb{R}^p$ adalah vektor parameter pada langkah iterasi ke-$i$. Untuk menghitung gradien, dibuat vektor acak $\Delta_i$ berukuran $p$, di mana setiap elemen $\Delta_{ij}$, $\forall$ $j\in {1,2,...,p}$, diambil secara seragam dari ${-1, 1}$. Selanjutnya, setiap elemen vektor acak $\Delta_i$ dikalikan dengan nilai kecil $c_i$ untuk membuat perturbasi acak. Gradien kemudian diestimasi sebagai

$$[\nabla f(x_i)]_j \approx \frac{f(x_i + c_i \Delta_i) - f(x_i - c_i \Delta_i)}{2c_i\Delta_{ij}}.$$

Secara intuitif, karena perturbasi acak diterapkan selama estimasi gradien, diharapkan bahwa deviasi kecil pada nilai eksak $f$ yang berasal dari noise dapat ditoleransi dan diperhitungkan. Bahkan, SPSA khususnya dikenal sangat tahan terhadap noise, dan hanya memerlukan dua pemanggilan hardware untuk setiap iterasi. Oleh karena itu, ini adalah salah satu optimizer yang sangat disukai untuk mengimplementasikan algoritma variasional.

Dalam tutorial ini, hyperparameter untuk iterasi ke-$i$, $a_i$ dan $c_i$, dihitung sebagai

$$a_i = \frac{a}{(A + i + 1)^\alpha} \quad \text{and} \quad c_i = \frac{c}{(i+1)^\gamma},$$

di mana nilai konstantanya adalah $A = 30$, $\alpha = 0.9$, $a = 0.3$, $c = 0.1$, dan $\gamma = 0.4$. Nilai-nilai ini dipilih dari [\[2\]](#references). Penyetelan hyperparameter yang tepat diperlukan untuk mendapatkan performa yang baik dari SPSA.

In [4]:
def spsa(
    fun, x0, args=(), A=30, alpha=0.9, a=0.3, c=0.1, gamma=0.4, maxiter=100
):
    nparams = len(x0)
    x = np.copy(x0)

    for i in range(maxiter):
        a_i = a / (A + i + 1) ** alpha
        c_i = c / (i + 1) ** gamma
        delta_i = np.random.choice([-1, 1], nparams)

        # two hardware calls
        eval_1 = fun(x + c_i * delta_i, *args)
        eval_2 = fun(x - c_i * delta_i, *args)

        # compute the gradient and update the parameters
        grad = (eval_1 - eval_2) / (2 * c_i) * np.reciprocal(delta_i)
        x = x - a_i * grad

    return x

In [8]:
def cost_func(
    params: Sequence,
    ansatz: QuantumCircuit,
    hamiltonian: SparsePauliOp,
    estimator: BaseEstimatorV2,
    cost_history_dict: dict,
) -> float:
    """Ground state energy evaluation."""
    energy = (
        estimator.run([(ansatz, hamiltonian, [params])]).result()[0].data.evs
    )

    cost_history_dict["iters"] += 1
    cost_history_dict["prev_vector"] = list(params)
    cost_history_dict["cost_history"].append(float(energy[0]))

    print(
        f"Fx Iters. done: {cost_history_dict['iters']} [Current cost: {round(energy[0], 5)}]",
        end="\r",
    )

    return energy


def solve(x0, isa_ansatz, isa_observable, maxiter=150):
    cost_history_dict = {
        "prev_vector": None,
        "iters": 0,
        "cost_history": [],
        "y_min": None,
    }

    # Evaluate the problem using a QPU via Qiskit IBM Runtime
    with Session(backend=backend) as session:
        estimator = EstimatorV2(mode=session)
        estimator.skip_transpilation = True
        estimator.options.environment.job_tags = ["TUT_HSVQE"]
        x_opt = spsa(
            cost_func,
            x0=x0,
            args=(isa_ansatz, isa_observable, estimator, cost_history_dict),
            maxiter=maxiter,
        )

        y_min = cost_func(
            x_opt, isa_ansatz, isa_observable, estimator, cost_history_dict
        )

    return y_min, cost_history_dict

In [9]:
np.random.seed(42)
num_params = ansatz.num_parameters
params = 2 * np.pi * np.random.random(num_params)

Di sini kita mengatur `maxiter = 50`. Perlu diperhatikan bahwa karena setiap iterasi memerlukan dua pemanggilan fungsi untuk menghitung gradien, total jumlah pemanggilan fungsi akan menjadi $2 \times \text{maxiter}$. Nilai `maxiter` bisa ditingkatkan ke nilai yang lebih tinggi untuk estimasi energi yang lebih baik.

In [10]:
maxiter = 50
spsa_min, spsa_history = solve(
    params, isa_ansatz, isa_observable, maxiter=maxiter
)

Fx Iters. done: 101 [Current cost: -3.03843]

### Step 4: Post-process and return result in desired classical format

* Input: Ground-state energy estimates during optimization
* Output: Estimated ground-state energy

In [11]:
print(f"Estimated ground state energy: {spsa_min}")

Estimated ground state energy: [-3.03842968]


In [12]:
results = {
    "spsa": spsa_history,
}

visualize_results(spsa_history)

<Image src="../docs/images/tutorials/spin-chain-vqe/extracted-outputs/ecd7762a-0.avif" alt="Output of the previous code cell" />

## Large-scale hardware example

A large-scale hardware example is not included in this tutorial. As the number of qubits increases, VQE encounters significant challenges due to the [barren plateau](/learning/courses/variational-algorithm-design/optimization-loops#barren-plateaus) phenomenon: the gradient of the cost function vanishes exponentially with system size, making optimization practically infeasible for large circuits. Combined with hardware noise, this means that scaling VQE to larger spin chains does not produce reliably reproducible results. For approaches that overcome these limitations, see the Next Steps section below.

## Challenge

Now that you have a working VQE implementation for the Heisenberg chain, try the following:

1. **Experiment with ansatz depth:** Modify the `reps` parameter in `efficient_su2` (for example, try `reps=1` and `reps=3`). How does ansatz depth affect the estimated ground-state energy and convergence speed? At what point do you observe diminishing returns or instability?
2. **Tune SPSA hyperparameters:** Adjust the learning rate schedule parameters (`a`, `c`, `alpha`, `gamma`, `A`) and observe how they impact convergence. Can you find a configuration that converges faster than the defaults used here?
3. **Compare coupling topologies:** Instead of using the backend's native coupling map, try constructing a simple nearest-neighbor linear chain and compare the results. How does the connectivity of the physical hardware affect the transpiled circuit depth and final energy estimate?

## References

\[1] Spall, J. C. (2002). Implementation of the simultaneous perturbation algorithm for stochastic optimization.
IEEE Transactions on Aerospace and Electronic Systems, 34(3), 817-823.

\[2] Sahin, M. Emre, et al. (2025). Qiskit Machine Learning: an open-source library for quantum machine learning tasks at scale on quantum hardware and classical simulators. arXiv:2505.17756.

## Next steps

<Admonition type="tip" title="Recommendations">
  If you found this work interesting, you might be interested in the following material:

  * **Try Sample-based Quantum Diagonalization (SQD):** As demonstrated in this tutorial, VQE faces challenges at scale due to barren plateaus and high measurement overhead. IBM has developed [Sample-based Quantum Diagonalization (SQD)](https://www.ibm.com/quantum/blog/quantum-diagonalization) as a more scalable alternative. Unlike VQE, SQD avoids variational optimization entirely; instead, a quantum computer generates samples and a classical computer projects the Hamiltonian onto a subspace spanned by those samples and diagonalizes it. This provides an upper bound to the ground-state energy with significantly fewer measurements and without susceptibility to barren plateaus. Follow the [SQD tutorial](/docs/tutorials/sample-based-quantum-diagonalization) to see this approach in action.
  * **Explore the Quantum Diagonalization Algorithms course:** Deepen your understanding of both VQE and SQD, including their trade-offs, in the [Quantum diagonalization algorithms](/learning/courses/quantum-diagonalization-algorithms) course on IBM Quantum Learning.
</Admonition>